In [ ]:
from google.colab import drive
import torch
import sys
import numpy as np
import math
import time
import os
import torch.nn.functional as F
from transformers import get_cosine_schedule_with_warmup
drive.mount('/content/gdrive', force_remount=True)


base_dir = "/content/gdrive/MyDrive/Junior/Second Semester/CS 6787 - Advanced ML Systems/regen-gpt2"
# base_dir = "/content/gdrive/MyDrive/Final Project"
sys.path.append(base_dir)

device = torch.device("cuda" if torch.cuda.is_available()
                else ("mps" if torch.backends.mps.is_available()
                else "cpu"))
#device = torch.device("meta")
print(F"Device set to {device}")


Mounted at /content/gdrive
Device set to cuda


In [ ]:
from Models.GPT2_Baseline import GPT2_Baseline, GPT2Config
from Datasets.DataLoader import CombinedBinDataLoader

In [ ]:
import gc
try:
    del model, optimizer, scheduler, train_loader, val_loader
except: pass

torch.cuda.empty_cache()
gc.collect()

180

In [ ]:
batch_per_iter = 8 # Adjust batch size based on your Colab GPU memory
grad_acc_factor = 16
eff_batch_size = batch_per_iter * grad_acc_factor
block_size = 1024
warmup_steps = 500
num_steps_train = 6866
weight_decay = .1
dropout = 0.1
lr = 3e-4
min_lr = .1 * lr
tokens_per_step = eff_batch_size * block_size

config = GPT2Config(num_heads = 12,
  num_layers = 12,
  vocab_size = 50257,
  embedding_dim = 768,
  block_size = block_size,
  dropout = dropout,
  weight_decay = weight_decay,
  pad_token_id = 50256)

model = GPT2_Baseline(config, device)
model = model.to(device)
model = torch.compile(model)

optimizer = torch.optim.AdamW(model.parameters(),
                              lr=lr,
                              betas=(0.9, 0.95),
                              weight_decay=weight_decay)

def get_lr(step):
    # warmup
    if step < warmup_steps:
        return lr * step / warmup_steps
    # cosine decay to min_lr
    progress = (step - warmup_steps) / (num_steps_train - warmup_steps)
    cosine   = 0.5 * (1 + math.cos(math.pi * progress))
    return min_lr + cosine * (lr - min_lr)

scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer, lambda step: get_lr(step) / lr
)
# scheduler = get_cosine_schedule_with_warmup(
#     optimizer,
#     num_warmup_steps=warmup_steps,
#     num_training_steps=num_steps_train
# )
scaler = torch.cuda.amp.GradScaler()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/tmp/ipykernel_44434/1285778670.py:45: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [ ]:
!cp "/content/gdrive/MyDrive/Junior/Second Semester/CS 6787 - Advanced ML Systems/regen-gpt2/Datasets/fineweb_1B.bin" "/content/fineweb_1B.bin"
train_loader, val_loader = CombinedBinDataLoader.create_loaders(
    '/content/fineweb_1B.bin', batch_per_iter, block_size, config, seed=0
)

Initialized loader with 109,863 chunks of size 8193.
Initialized loader with 12,207 chunks of size 8193.


In [ ]:
message = model.infer("Hi, my name is John. It is nice", 30, 1, 50)
print(message)

Hi, my name is John. It is nicevis coins Maggie Soldiers yetGR capital correspondingutigeries Recon chemistry "+ heroes M uterus marrying mobile presented ba45107 shouted replacementMobBear 263utz TTLYesterday


In [ ]:

@torch.no_grad()
def estimate_loss(model, loader, device, eval_iters=10):
    model.eval()

    losses = []

    for _ in range(eval_iters):
        x, y, _ = loader.get_data()
        x, y = x.to(device), y.to(device)

        _, loss = model(x, y)
        losses.append(loss.item())

    model.train()

    avg = sum(losses) / len(losses)
    ppl = torch.exp(torch.tensor(avg)).item()

    return avg, ppl



def train_loop(model, optimizer, scheduler, device, train_loader, val_loader,
               num_steps_train, num_steps_val):

    print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

    loss, ppl= estimate_loss(model, val_loader, device, num_steps_val)
    print(f"Step    0 | Val Loss: {loss:.4f} | PPL: {ppl:.2f}")
    start = time.time()
    tokens_seen = 0

    #Define some useful constants
    for step in range(num_steps_train):
        if step % 100 == 0 and step > 0:
            loss, ppl = estimate_loss(model, val_loader, device, num_steps_val)
            print(f"Step {step:4d} | Val Loss: {loss:.4f} | PPL: {ppl:.2f}")
            if device == 'mps':
                torch.mps.empty_cache()


        avg_loss = 0
        optimizer.zero_grad(set_to_none=True)
        for _ in range(grad_acc_factor):
            x, y, _ = train_loader.get_data()
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)


            with torch.autocast(device_type=device.type, dtype=torch.float16):
                _, loss = model(x, y)
                loss /= grad_acc_factor
                avg_loss += loss.item()

            scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        tokens_seen += tokens_per_step
        if step % 10 == 0:
            stop = time.time()
            print(f"step {step:4d}/{num_steps_train} | tokens {tokens_seen:,} | Train loss {avg_loss:.4f} | Time Since Last Train Print {(stop-start):.4f} seconds")
            start = time.time()

        if(step % 1000 == 0 and step > 0):
            path = "/content/gdrive/MyDrive/Junior/Second Semester/CS 6787 - Advanced ML Systems/regen-gpt2/ckpt_baseline_step_{step}.pt".format(step=step)
            torch.save({
            "step": step,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "loss": loss.item(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict()
        }, path)



In [ ]:
torch.cuda.empty_cache()
train_loop(model, optimizer, scheduler, device, train_loader, val_loader, num_steps_train, 5*grad_acc_factor)

Trainable parameters: 124,439,808
Step    0 | Val Loss: 11.0214 | PPL: 61168.64
step    0/6866 | tokens 131,072 | Train loss 11.0207 | Time Since Last Train Print 9.8523 seconds
step   10/6866 | tokens 1,441,792 | Train loss 10.8541 | Time Since Last Train Print 78.2793 seconds
step   20/6866 | tokens 2,752,512 | Train loss 10.3776 | Time Since Last Train Print 78.1241 seconds
step   30/6866 | tokens 4,063,232 | Train loss 9.8941 | Time Since Last Train Print 78.1262 seconds
step   40/6866 | tokens 5,373,952 | Train loss 9.5401 | Time Since Last Train Print 78.1214 seconds
step   50/6866 | tokens 6,684,672 | Train loss 9.2825 | Time Since Last Train Print 77.6579 seconds
step   60/6866 | tokens 7,995,392 | Train loss 9.0063 | Time Since Last Train Print 77.6689 seconds
step   70/6866 | tokens 9,306,112 | Train loss 8.7648 | Time Since Last Train Print 77.4723 seconds
step   80/6866 | tokens 10,616,832 | Train loss 8.4855 | Time Since Last Train Print 77.2006 seconds
step   90/6866 | to

In [ ]:
message = model.infer("""Ancient Rome (753 BC–AD 476) evolved from a small Italian city-state into a massive Mediterranean empire, structured into three main periods: the Monarchy/Kingdom (753–509 BC), the Republic (509–27 BC), and the Empire (27 BC–AD 476).
It became a dominant power under emperors like Augustus, falling in the West due to internal instability and external pressures, while the Eastern Byzantine Empire continued until 1453.
""", 100, .8, 50)
print(message)

Ancient Rome (753 BC–AD 476) evolved from a small Italian city-state into a massive Mediterranean empire, structured into three main periods: the Monarchy/Kingdom (753–509 BC), the Republic (509–27 BC), and the Empire (27 BC–AD 476).
It became a dominant power under emperors like Augustus, falling in the West due to internal instability and external pressures, while the Eastern Byzantine Empire continued until 1453.
The second in the world were that the state was the oldest-style state in the early series, which appears to be the part of the World Africa. The US War was a significant factor of the first-term country for the state of the Middle. The British was the capital to the World and the World. The government was also a good, but the latter is the most recent economy of the United States.
The New York National Court (19th) was a member of the South and


In [ ]:
path = "/content/gdrive/MyDrive/Junior/Second Semester/CS 6787 - Advanced ML Systems/regen-gpt2/ckpt_baseline_step_6866.pt"
torch.save({
            "step": 6866,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict()
        }, path)

In [ ]:
loss_fwd, ppl = estimate_loss(model, val_loader, device, 5*grad_acc_factor)
print(f"Val FWD: {loss_fwd:.4f} | PPL: {ppl:.2f}")

Val FWD: 5.5937 | PPL: 268.73
